# DiodeMCP Track A — `01_dataset_factory.ipynb`
**Colab Pro+ (`L4 24GB` / `A100 40–80GB`) Matched-Generator Dataset Factory**

Produces:
- **`diodebench_mcp_250k`** (`240,000` train + `10,000` `val_calib`): `85k` programmatic T1–T7 mutators + `165k` matched-generator `Qwen/Qwen2.5-14B-Instruct-AWQ` attack paraphrases & Trifecta-safe benign developer twins (`I9`), with `25%` `apply_missing_intent_mask()` and symmetric `20%` `FlipJev` (`I7`).
- **`trifecta_dev_bench_5k`** (`5,000` `eval_holdout`): category-stratified, strictly group-disjoint developer Trifecta workflows + held-out `AgentDojo` & `MCPSecBench` evaluation adapters.
- **3 Backbone Packed Parquet Suites**: `answerdotai/ModernBERT-base`, `answerdotai/ModernBERT-large`, and `Qwen/Qwen3-0.6B` under `/content/drive/MyDrive/diodemcp/datasets/` with zero-leakage & zero-real-secret verification (`I10`).

In [ ]:
# 1. Mount Google Drive, resolve model-lab checkout, & install model-lab + vLLM
import json
import os
import subprocess
from datetime import datetime, timezone
from pathlib import Path

try:
    from google.colab import drive  # type: ignore[import-not-found]
    drive.mount("/content/drive", force_remount=False)
except ImportError:
    print("Running outside Google Colab runtime; skipping drive.mount().")

!nvidia-smi
!pip install -q uv

CANDIDATE_MODEL_LAB_DIRS = (
    Path("/content/diodemcp/model-lab"),
    Path("/content/drive/MyDrive/diodemcp/model-lab"),
    Path("./model-lab"),
    Path("../model-lab"),
    Path("."),
)
MODEL_LAB_DIR = next(
    (
        cand.resolve()
        for cand in CANDIDATE_MODEL_LAB_DIRS
        if (cand / "pyproject.toml").is_file() and (cand / "src" / "diodemodel").is_dir()
    ),
    None,
)
if MODEL_LAB_DIR is None:
    raise FileNotFoundError(
        "Could not locate model-lab/pyproject.toml in any candidate path: "
        f"{[str(c) for c in CANDIDATE_MODEL_LAB_DIRS]}. "
        "Clone diodemcp to /content/diodemcp or sync model-lab to /content/drive/MyDrive/diodemcp/model-lab."
    )

print(f"Installing model-lab from: {MODEL_LAB_DIR}")
subprocess.run(["uv", "pip", "install", "--system", "-e", str(MODEL_LAB_DIR), "vllm"], check=True)

In [ ]:
# 2. Configure Drive workspace paths, raw_shards directory, and live telemetry logger
DRIVE_ROOT = Path(os.environ.get("DIODEMCP_DRIVE_ROOT", "/content/drive/MyDrive/diodemcp"))
DATASETS_DIR = DRIVE_ROOT / "datasets"
TOKENIZERS_DIR = DATASETS_DIR / "tokenizers"
SHARDS_DIR = DATASETS_DIR / "raw_shards"
STATUS_LOG_PATH = DATASETS_DIR / "live_status.jsonl"

for directory in (DATASETS_DIR, TOKENIZERS_DIR, SHARDS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

SMOKE_MODE = os.environ.get("DIODEMCP_SMOKE_MODE", "0") == "1"
SYNTH_MODEL_ID = "Qwen/Qwen2.5-14B-Instruct-AWQ"
N_TRAIN = 240 if SMOKE_MODE else 240_000
N_VAL_CALIB = 40 if SMOKE_MODE else 10_000
N_EVAL_HOLDOUT = 60 if SMOKE_MODE else 5_000
SHARD_SIZE = 60 if SMOKE_MODE else 2_500
RNG_SEED = 42


def log_factory_event(stage: str, **payload: object) -> None:
    record = {
        "ts": datetime.now(timezone.utc).isoformat(),
        "notebook": "01_dataset_factory",
        "stage": stage,
        **payload,
    }
    with STATUS_LOG_PATH.open("a", encoding="utf-8") as fh:
        fh.write(json.dumps(record, sort_keys=True) + "\n")
        fh.flush()
    print(f"[{stage}]", json.dumps(payload, sort_keys=True))


log_factory_event(
    "factory_initialized",
    smoke_mode=SMOKE_MODE,
    shards_dir=str(SHARDS_DIR),
    n_train=N_TRAIN,
    n_val_calib=N_VAL_CALIB,
    n_eval_holdout=N_EVAL_HOLDOUT,
    synth_model_id=SYNTH_MODEL_ID,
)

In [ ]:
# 3. Build all 3 normative Jev tokenizers (12 special tokens + C2 structural ID map)
from tokenizers import Tokenizer

from diodemodel.contract import BundleManifest
from diodemodel.tokenizer_builder import build_jev_tokenizer

BACKBONE_SPECS = {
    "modernbert_base": ("answerdotai/ModernBERT-base", "diodemcp-jev-base-int8"),
    "modernbert_large": ("answerdotai/ModernBERT-large", "diodemcp-jev-large-int8"),
    "qwen3_0_6b": ("Qwen/Qwen3-0.6B", "diodemcp-nanojev-0.6b"),
}

built_tokenizers: dict[str, tuple[Tokenizer, BundleManifest]] = {}
for profile_key, (hf_model_id, bundle_model_id) in BACKBONE_SPECS.items():
    out_tok_dir = TOKENIZERS_DIR / profile_key
    special_ids = build_jev_tokenizer(hf_model_id, out_tok_dir)
    rust_tok = Tokenizer.from_file(str(out_tok_dir / "tokenizer.json"))
    manifest = BundleManifest(
        model_id=bundle_model_id,
        sha256_model_onnx="0" * 64,
        sha256_tokenizer_json="0" * 64,
        sha256_calibration_json="0" * 64,
        special_token_ids=special_ids,
        ece_val=0.015,
    )
    built_tokenizers[profile_key] = (rust_tok, manifest)
    log_factory_event(
        "tokenizer_ready",
        profile=profile_key,
        hf_model_id=hf_model_id,
        vocab_size=rust_tok.get_vocab_size(),
    )

In [ ]:
# 4. Ingest permissive seeds & stratify disjoint repo groups across T1-T7
import pyarrow.parquet as pq

from diodemodel.dataset_packer import partition_seeds_stratified_by_group
from diodemodel.parse_seeds import SeedRecord, load_permissive_seeds

SEEDS_PARQUET_PATH = DATASETS_DIR / "permissive_seeds.parquet"
load_permissive_seeds(SEEDS_PARQUET_PATH, include_remote=not SMOKE_MODE)
all_seeds = [SeedRecord.model_validate(r) for r in pq.read_table(SEEDS_PARQUET_PATH).to_pylist()]

partition = partition_seeds_stratified_by_group(all_seeds, rng_seed=RNG_SEED)
train_seeds = partition.train_seeds
val_seeds = partition.val_seeds
eval_dev_seeds = partition.eval_dev_seeds
holdout_bench_seeds = partition.holdout_bench_seeds

log_factory_event(
    "seeds_partitioned",
    total_seeds=len(all_seeds),
    train_groups=len(partition.train_groups),
    val_groups=len(partition.val_groups),
    eval_dev_groups=len(partition.eval_dev_groups),
    holdout_bench_seeds=len(holdout_bench_seeds),
)

In [ ]:
# 5. Synthesize 85k programmatic + 165k matched-generator vLLM shards (single AWQ FP16 engine, resumable)
from diodemodel.dataset_packer import adapt_benchmark_holdout_seeds
from diodemodel.mutators import JevTrainingPair, generate_programmatic_pairs
from diodemodel.synth_vllm import (
    get_or_create_vllm_completion_fn,
    load_synthesized_shards,
    synthesize_matched_shards,
)

shared_completion_fn = (
    None
    if SMOKE_MODE
    else get_or_create_vllm_completion_fn(model_id=SYNTH_MODEL_ID)
)


def build_resumable_split(
    split_name: str,
    pool_seeds: list[SeedRecord],
    total_target: int,
    seed_offset: int,
) -> list[JevTrainingPair]:
    # Exact 85k / 165k (34% programmatic / 66% matched-generator vLLM) split across 250k
    n_prog = round(total_target * 0.34)
    n_vllm = total_target - n_prog

    prog_pairs = list(
        generate_programmatic_pairs(
            pool_seeds,
            n_samples=n_prog,
            rng_seed=RNG_SEED ^ seed_offset,
        )
    )
    shard_dir = SHARDS_DIR / split_name
    shard_paths = synthesize_matched_shards(
        seeds=pool_seeds,
        out_shard_dir=shard_dir,
        status_log_path=STATUS_LOG_PATH,
        n_samples=n_vllm,
        shard_size=SHARD_SIZE,
        rng_seed=RNG_SEED ^ (seed_offset + 17),
        offline=SMOKE_MODE,
        model_id=SYNTH_MODEL_ID,
        completion_fn=shared_completion_fn,
    )
    vllm_pairs = list(load_synthesized_shards(shard_paths, max_rows=n_vllm))
    merged = [
        p.model_copy(update={"pair_id": f"{split_name}_{idx:06d}_{p.category.lower()}"})
        for idx, p in enumerate([*prog_pairs, *vllm_pairs])
    ]
    log_factory_event(
        "split_synthesized",
        split=split_name,
        programmatic=len(prog_pairs),
        vllm=len(vllm_pairs),
        total=len(merged),
    )
    return merged


# diodebench_mcp_250k = 240,000 train + 10,000 val_calib (85,000 prog + 165,000 vLLM total)
train_pairs = build_resumable_split("train", train_seeds, N_TRAIN, 0x1001)
val_calib_pairs = build_resumable_split("val_calib", val_seeds, N_VAL_CALIB, 0x2002)

# trifecta_dev_bench_5k = held-out AgentDojo & MCPSecBench + disjoint Trifecta dev workflows
n_bench_eval = max(2, min(round(N_EVAL_HOLDOUT * 0.40), N_EVAL_HOLDOUT - 1))
bench_eval_pairs = adapt_benchmark_holdout_seeds(
    holdout_bench_seeds,
    target_count=n_bench_eval,
    rng_seed=RNG_SEED ^ 0x3003,
)
dev_eval_pairs = build_resumable_split(
    "eval_trifecta",
    eval_dev_seeds,
    N_EVAL_HOLDOUT - len(bench_eval_pairs),
    0x4004,
)
eval_holdout_pairs = [*bench_eval_pairs, *dev_eval_pairs]

splits = {
    "train": train_pairs,
    "val_calib": val_calib_pairs,
    "eval_holdout": eval_holdout_pairs,
}

In [ ]:
# 6. Stream-pack diodebench_mcp_250k & trifecta_dev_bench_5k Parquet tables (10k/batch) + zero-leakage audit
from diodemodel.dataset_packer import audit_packed_splits, pack_dataset_to_parquet

for profile_key, (rust_tok, manifest) in built_tokenizers.items():
    profile_out_dir = DATASETS_DIR / profile_key
    parquet_paths = pack_dataset_to_parquet(
        splits=splits,
        tokenizer=rust_tok,
        manifest=manifest,
        out_dir=profile_out_dir,
        max_bucket=4096,
        batch_size=10_000,
    )
    audit = audit_packed_splits(splits, parquet_paths)
    log_factory_event(
        "parquet_packed_and_audited",
        profile=profile_key,
        out_dir=str(profile_out_dir),
        total_packed_rows=audit.total_packed_rows,
        leakage_count=audit.leakage_count,
        real_secret_hits=audit.real_secret_hits,
        split_counts=audit.split_counts,
        files={k: str(v) for k, v in parquet_paths.items()},
    )

print("Dataset factory complete: diodebench_mcp_250k + trifecta_dev_bench_5k packed & verified (0 leakage).")